# MOLDR: 小さな分子から分解・再結合・E2E を試す

最初に動作を確かめたい方向けのチュートリアルです。8 個のサンプル分子を入力して、
**頻出部分構造の抽出 → 再結合候補の表示 → E2E 環境での分子生成 → 結果の保存**まで進めます。
最後に、同じ設定を使った短い PPO 学習と checkpoint からの生成を任意で試せます。

通常の「Run All」は CPU のみで動き、学習や Ray の起動、外部データのダウンロードは行いません。
既存の `models/model_300dim.pkl` を使用します。後半の `RUN_TRAINING = True` を選んだ場合のみ、
ローカル Ray と CPU で PPO を 1 iteration 実行します。W&B のアカウントは不要です。

**対応する実装:** この Notebook は `moldr.factory` / `moldr.cli` と E2E v2 を含む開発版用です。
2026-11-07 時点では、この共通 CLI は `main` および PR #1 の先端にはまだ含まれていません。
Notebook と同じソース一式を使ってください。旧 legacy / E2E v1 の checkpoint を E2E v2 に
そのまま読み替えることはできません。

## 1. 実行環境と出力先

README の Installation を実行し、このプロジェクトの Python 環境を Jupyter のカーネルに選びます。
以下はリポジトリ直下、または `examples/` から実行できます。最初に Python のパスを表示します。

出力は毎回新しい `outputs/tutorials/quickstart_.../` に保存します。
元のデータや既存の学習結果を上書きしません。

In [ ]:
import io
import json
import sys
from contextlib import redirect_stdout
from pathlib import Path
from tempfile import mkdtemp

ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "moldr").is_dir() and (p / "pyproject.toml").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError("リポジトリ直下か examples/ から Notebook を開いてください。")
if not (ROOT / "moldr/factory.py").is_file():
    raise RuntimeError("この Notebook と同じ開発版のソースが必要です。")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import yaml
from IPython.display import display
from rdkit import Chem, rdBase
from rdkit.Chem import Draw, QED

from moldr.chemutils import get_mol, get_smiles, validate_reassembly_mol

model_path = ROOT / "models/model_300dim.pkl"
if not model_path.is_file():
    raise FileNotFoundError(f"付属の Mol2Vec モデルが見つかりません: {model_path}")
output_parent = ROOT / "outputs/tutorials"
output_parent.mkdir(parents=True, exist_ok=True)
OUTPUT = Path(mkdtemp(prefix="quickstart_", dir=output_parent))
SEED = 7
print("Python:", sys.executable)
print("Repository:", ROOT)
print("Output:", OUTPUT)

## 2. サンプルデータを見る

ここでは説明用に選んだ 8 個の小さな分子を使います。実験用データセットの抜粋ではありません。
`smiles` が入力形式で、1 行が 1 分子です。自分のデータに置き換える場合は、まずこの程度の件数で試してください。

QED はこの例で使う分子スコアです。値域は 0〜1 ですが、この値だけで研究目的への適合性は判断できません。

In [ ]:
sample = pd.DataFrame({
    "name": ["ethanol", "propanol", "methoxypropane", "methoxyethane",
             "ethylamine", "propylamine", "acetic acid", "ethyl acetate"],
    "smiles": ["CCO", "CCCO", "CCCOC", "CCOC", "CCN", "CCCN",
               "CC(=O)O", "CCOC(=O)C"],
})
input_mols = [get_mol(s) for s in sample["smiles"]]
assert all(m is not None for m in input_mols), "入力に無効な SMILES があります。"
for mol in input_mols:
    validate_reassembly_mol(mol)
sample["qed"] = [QED.qed(m) for m in input_mols]
sample.to_csv(OUTPUT / "input_molecules.csv", index=False)
display(sample)
display(Draw.MolsToGridImage(
    input_mols, legends=sample["name"].tolist(), molsPerRow=4,
    subImgSize=(220, 160), useSVG=True,
))

## 3. gSpan で building block を抽出する

`raw` は分子を原子と結合のグラフとして扱います。ここでは **2 分子以上に現れる、2〜4 原子の部分構造**を探します。
`support` は割合ではなく最小出現分子数です。`jt` による環・クリークを考慮した分解は既存の
[decomposition.ipynb](decomposition.ipynb) を参照してください。

抽出した分子を検証・重複除去し、`building_blocks.csv` に書き出します。
このファイルは、後で共通 factory / CLI の `fragments.path` にそのまま渡せます。
CSV の先頭列はインデックス、次の列は SMILES です。

In [ ]:
from moldr.core.decompose import DefaultConfig, MolsMining

mining_config = DefaultConfig(
    data_path=OUTPUT / "toy.data", support=2, lower=2, upper=4,
    method="raw", is_save=False,
)
miner = MolsMining(mining_config)
gspan = miner.decompose(input_mols)
mined_mols = miner.gspan_to_mols(gspan, smiles_list=sample["smiles"].tolist())

fragment_smiles = set()
for mol in mined_mols:
    if mol is None:
        continue
    canonical = Chem.MolToSmiles(mol)
    parsed = get_mol(canonical)
    if parsed is None:
        continue
    validate_reassembly_mol(parsed)
    fragment_smiles.add(get_smiles(parsed))
fragment_smiles = sorted(fragment_smiles)
assert fragment_smiles, "部分構造がありません。support や原子数範囲を見直してください。"
fragments = pd.DataFrame({"smiles": fragment_smiles})
fragment_path = OUTPUT / "building_blocks.csv"
fragments.to_csv(fragment_path)  # The loader expects an index column.
print(f"入力: {len(input_mols)} 分子 → building blocks: {len(fragments)} 個")
display(fragments)
display(Draw.MolsToGridImage(
    [get_mol(s) for s in fragment_smiles], legends=fragment_smiles,
    molsPerRow=5, subImgSize=(180, 140), useSVG=True,
))

このサンプルと設定では、`CC`、`CO`、`CCO` など 13 個の building block が得られます。
入力分子、`support`、原子数の範囲を変えると、個数と内容も変わります。
本格的なデータを使う前に、抽出結果が意図した部分構造になっているか図で確認してください。

## 4. 再結合候補を表示する

`CCO` に、先ほど得た `CC(=O)O` を再結合します。原子や環の辺を共有する結合方法を列挙するため、
単に 2 つの SMILES をつなぐ操作ではありません。

legacy と E2E の両方が使う `CandidateGenerator` で候補を作り、QED と構造を比較します。
このセルではすべての候補を説明用に採点します。E2E の通常の遷移では、選択した候補だけを採点します。

In [ ]:
from moldr.chemistry import CandidateGenerator

base_smiles = "CCO"
attach_smiles = "CC(=O)O"
assert attach_smiles in fragment_smiles
with rdBase.BlockLogs():  # Invalid intermediate merges are discarded by the generator.
    candidates = CandidateGenerator().generate(get_mol(base_smiles), get_mol(attach_smiles))
assert candidates, "有効な再結合候補が得られませんでした。"
candidate_table = pd.DataFrame({
    "candidate": range(len(candidates)),
    "smiles": [c.smiles for c in candidates],
    "qed": [QED.qed(c.mol) for c in candidates],
})
candidate_table.to_csv(OUTPUT / "reassembly_candidates.csv", index=False)
display(candidate_table)
display(Draw.MolsToGridImage(
    [c.mol for c in candidates[:12]],
    legends=[f"{i}: QED={q:.3f}" for i, q in enumerate(candidate_table["qed"][:12])],
    molsPerRow=4, subImgSize=(220, 170), useSVG=True,
))

## 5. E2E 環境を 1 エピソード動かす

E2E では **A: building block の選択 → B: できあがった再結合候補の選択**を繰り返します。
legacy は A だけをエージェントが選び、B は環境がスコア最大の候補を選びます。
両者は行動・観測の形式が異なるため、学習済み重みをそのまま交換することはできません。

ここでは学習済みモデルを使わず、各段階で `action_mask` が 1 の選択肢から一様ランダムに選びます。
小さな環境の動作確認が目的で、最適化性能の評価ではありません。
Mol2Vec は付属の固定モデルを使い、分子表現を計算します。

最大 8 decisions（通常は 4 回の再結合）で止めます。
`score_threshold=1.1` は QED で早期終了させないための設定です。
`atom_limit` は `>` で終了判定するため、最終分子がこの数を超える場合があります。

In [ ]:
from moldr.experiment import load_config
from moldr.factory import make_environment, resolve_config

config = load_config(cli_values={
    "environment.mode": "e2e",
    "environment.base_smiles": "C",
    "environment.atom_limit": 12,
    "environment.max_episode_steps": 8,
    "environment.max_attach": 32,
    "environment.score_threshold": 1.1,
    "environment.reward": "improvement",
    "fragments.path": str(fragment_path),
    "representation.model_path": str(model_path),
    "scoring.objective": "QED",
    "training.seed": SEED,
    "training.iterations": 1,
    "training.num_workers": 0,
    "training.num_cpus": 2,
    "training.num_gpus": 0.0,
    "training.train_batch_size": 64,
    "training.minibatch_size": 32,
    "training.num_epochs": 1,
    "training.hidden": 32,
    "evaluation.episodes": 3,
    "output.directory": str(OUTPUT / "ppo"),
    "output.logger": "local",
})
config, loaded_blocks, _ = resolve_config(config)
assert loaded_blocks == fragment_smiles
config_path = OUTPUT / "tutorial.yaml"
config_path.write_text(yaml.safe_dump(config.to_dict(), sort_keys=False), encoding="utf-8")
env = make_environment(config)
obs, info = env.reset(seed=SEED)
assert env.observation_space.contains(obs)
print("環境:", type(env).__name__, "/ observation version:", info["observation_version"])
print("観測キー:", sorted(obs))
print("有効な初期行動数:", int(obs["action_mask"].sum()))
print("初期 SMILES / QED:", env.base_smiles, round(info["score"], 4))

`action_mask` は段階ごとに変わります。A では building block の番号、B ではその時点の候補の番号です。
行動空間全体から無条件に乱数を選ぶと、無効な行動を選んでしまいます。

次の表の `phase` は**実行した行動の段階**、`smiles` と `qed` は**行動後の分子**です。
A では通常は分子とスコアが変わらず、B で変わります。
`reward` は通常 QED の差分で、終了条件によって完了報酬が加わる場合があります。
`terminated` は環境の終了条件、`truncated` は decisions の上限到達です。

In [ ]:
rng = np.random.default_rng(SEED)
rows = []
terminated = truncated = False
try:
    for decision in range(1, config.environment.max_episode_steps + 1):
        valid_actions = np.flatnonzero(obs["action_mask"] > 0)
        assert valid_actions.size, "終了前なのに有効な行動がありません。"
        action = int(rng.choice(valid_actions))
        with rdBase.BlockLogs():
            obs, reward, terminated, truncated, info = env.step(action)
        assert env.observation_space.contains(obs)
        rows.append({
            "decision": decision, "phase": info["phase"], "action": action,
            "valid_actions": len(valid_actions),
            "selected_block": info.get("chosen_bb_smiles", ""),
            "candidate_count": info.get("n_candidates"),
            "smiles": info["final_smiles"], "qed": env.prev_score,
            "reward": reward, "terminated": terminated, "truncated": truncated,
        })
        if terminated or truncated:
            break
    assert terminated or truncated, "エピソードが指定した上限内に終了していません。"
finally:
    env.close()
trajectory = pd.DataFrame(rows)
display(trajectory)

## 6. 結果を保存して確認する

`trajectory.csv` には各 decision の履歴、`episode.json` には最終分子と終了理由を保存します。
図には同じ SMILES を 1 回だけ表示します。乱数 seed を固定していますが、ライブラリや候補生成の変更で
候補順序が変われば、同じ seed でも軌跡は変わることがあります。

In [ ]:
trajectory.to_csv(OUTPUT / "trajectory.csv", index=False)
summary = {
    "policy": "uniform_valid_actions", "seed": SEED,
    "smiles": rows[-1]["smiles"], "qed": float(rows[-1]["qed"]),
    "total_reward": float(trajectory["reward"].sum()), "decisions": len(rows),
    "terminated": bool(terminated), "truncated": bool(truncated),
}
(OUTPUT / "episode.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
assert get_mol(summary["smiles"]) is not None
unique_states = trajectory.drop_duplicates("smiles")
display(Draw.MolsToGridImage(
    [get_mol(s) for s in unique_states["smiles"]],
    legends=[f"step {r.decision}, QED={r.qed:.3f}" for r in unique_states.itertuples()],
    molsPerRow=4, subImgSize=(220, 170), useSVG=True,
))
print(json.dumps(summary, indent=2))
print("保存ファイル:", ", ".join(p.name for p in sorted(OUTPUT.iterdir()) if p.is_file()))

## 7. 任意: 同じ設定で CPU の PPO 学習を試す

ここから先は `RUN_TRAINING = True` にすると動きます。
まず CLI の `--dry-run` で設定とアセットを検証します。これは通常の Run All でも実行します。

学習は 1 iteration、batch 64、更新 1 epoch、rollout worker 0、GPU 0 の小さな設定です。
worker 0 でも **ローカル Ray の起動が必要**です。ローカルプロセス・ソケットが利用できる環境で実行してください。
起動時間がかかるため、Notebook の最初の動作確認とは分けています。

この 1 iteration は学習・保存・復元の接続確認です。ランダム選択より良い分子を生成できる保証はありません。
`cli.main([...])` はターミナルの `python -m moldr.cli ...` と同じエントリーポイントです。

In [ ]:
from moldr.cli import main as moldr_cli

RUN_TRAINING = False
train_args = ["train", "--config", str(config_path)]
with redirect_stdout(io.StringIO()):
    checked_config = moldr_cli([*train_args, "--dry-run"])
print("CLI dry-run: OK; CPU / E2E / QED /", len(loaded_blocks), "building blocks")

training_run = None
if RUN_TRAINING:
    training_run = moldr_cli(train_args)
    print("学習結果:", training_run)
else:
    print("学習は未実行です。試す場合は RUN_TRAINING = True にして、このセルから再実行してください。")

## 8. 任意: checkpoint を復元して分子を生成する

上の学習を実行した場合、run ディレクトリに保存された設定とアセットを使って 3 エピソード生成します。
`--checkpoint` には run ディレクトリを渡せます。
観測次元や fragment 表を手作業で変更せず、保存時の構成から復元してください。

In [ ]:
if training_run is not None:
    evaluation_run = moldr_cli([
        "sample", "--checkpoint", str(training_run), "--episodes", "3",
    ])
    generated = pd.read_json(evaluation_run / "molecules.json")
    display(generated[["seed", "smiles", "score", "steps", "terminated", "truncated"]])
    display(Draw.MolsToGridImage(
        [get_mol(s) for s in generated["smiles"]],
        legends=[f"QED={s:.3f}" for s in generated["score"]],
        molsPerRow=3, subImgSize=(220, 170), useSVG=True,
    ))
    print("生成結果:", evaluation_run)
else:
    print("checkpoint からの生成は、上の学習を実行した後で試せます。")

## 研究用データへ置き換える場所

| 変更したいもの | 入口 | 確認すること |
| --- | --- | --- |
| 入力分子 | 手順 2 の `sample` | SMILES が解釈できること。小さな件数から試す |
| 分解の粒度 | 手順 3 の `support`, `lower`, `upper`, `method` | 頻出部分構造の数と内容 |
| 既存の building block 表 | 手順 5 の `fragments.path` | index 列付き CSV の次の列が SMILES。`.smi` / `.txt` は 1 行 1 SMILES も可 |
| スコア | `scoring.objective` | 登録済みの `QED`, `qed-sa`, `logp-8`, `logp-minus1`。変更時は表の `qed` という表示も見直す |
| 独自スコア | `moldr.scoring.register_objective` | 大きい値を良い候補として扱う。登録は環境作成や学習の前に行う |
| 初期分子・探索範囲 | `environment.base_smiles`, `atom_limit`, `max_episode_steps`, `max_attach` | 計算量と終了条件。`max_attach` を超える候補は切り詰められる |
| 学習量 | `training.iterations` と PPO の batch / epoch 設定 | 複数 seed、独立な評価条件、ランダム baseline と比較する |

付属の採掘済み表を試す場合は、例えば
`ROOT / "data/building_blocks/guacamol/guacamol_s10000_l2.csv"` を `fragments.path` に指定します。
これは 13 個の例より大きいため、計算量も増えます。

現在の再結合実装は、明示的な原子・結合の立体化学を扱えず、環境ではその入力を拒否します。
また、電荷・同位体の保持などの修正により、旧実装と生成結果が変わるケースがあります。
中性・立体化学なしのテストでの一致と、すべての入力・checkpoint の互換性は別に確認する必要があります。

詳細は [実験設定](../docs/experiment_configuration.md)、
[旧 API を含む説明](../README.md#faster-end-to-end-and-generative-extensions)、
[学習結果と限界](../docs/experiment_learning500.md) を参照してください。